In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import timm, types, sys, h5py, wandb, random
from pathlib import Path
from tqdm.auto import tqdm
from scipy.stats import spearmanr
from torch.utils.data import DataLoader, WeightedRandomSampler, TensorDataset
from peft import LoraConfig
from peft.tuners.lora import LoraModel
import torchvision.transforms as T

sys.path.append(".")
from src.dataset import HistologicalImageDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [3]:
DATA_DIR     = "/data/BREAKHIS"
DATASET_NAME = Path(DATA_DIR).name
BASE_CKPT    = Path("checkpoints")
BASE_CACHE   = Path("/data/data_cache")

CFG = dict(
    data_dir      = DATA_DIR,
    dataset_name  = DATASET_NAME,
    img_size      = 224,
    batch_size    = 64,
    num_workers   = 8,
    seed          = 42,
    output_dir    = BASE_CKPT / DATASET_NAME / "uni_finetuned",
    dataset_cache = BASE_CACHE / f"{DATASET_NAME}_forecaster_dataset.h5",
    forecaster_dir= BASE_CKPT / DATASET_NAME / "forecaster_curriculum",
    layer_target  = 23,
    layers_source = [2, 4],
    hidden        = 256,
    n_heads       = 4,
    n_layers      = 2,
    dropout       = 0.2,

    # ── Curriculum ──────────────────────────────────
    # Ispirato da Li et al. (NeurIPS 2024): i layer profondi
    # trasmettono informazione più utile per il task.
    # Partiamo da layer vicini all'output (task facile)
    # e spostiamo il target indietro progressivamente.
    #
    # Strategia:
    #   Fase 1: predict layer_target   dai layer vicini  (warm-up)
    #   Fase 2: predict layer_target   da layer_source   (target finale)
    #   Con warm-up dai layer intermedi il forecaster
    #   impara prima il "linguaggio" dell'attenzione matura,
    #   poi generalizza a distanze maggiori.
    curriculum_steps = [8, 15, 20],  # layer intermedi usati come target nel warm-up
    curriculum_epochs_per_step = 5,        # epoch per ciascun step del curriculum
    # ────────────────────────────────────────────────

    epochs        = 20,   # epoch finali sul target reale
    lr            = 1e-4,
    weight_decay  = 0.05,

    # Distillation loss weight (Li et al. usano λ=3 per attn distillation)
    lambda_kl     = 1.0,
    lambda_mse    = 0.1,

    wandb_project = "attention-forecaster-curriculum",
)

for k in ["output_dir", "dataset_cache", "forecaster_dir"]:
    CFG[k] = CFG[k] if isinstance(CFG[k], Path) else Path(CFG[k])
CFG["dataset_cache"].parent.mkdir(parents=True, exist_ok=True)
CFG["output_dir"].mkdir(parents=True, exist_ok=True)
CFG["forecaster_dir"].mkdir(parents=True, exist_ok=True)

torch.manual_seed(CFG["seed"])
torch.cuda.manual_seed_all(CFG["seed"])
np.random.seed(CFG["seed"])
random.seed(CFG["seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
print(f"Dataset: {DATASET_NAME}")

Dataset: BREAKHIS


In [4]:
train_tf = T.Compose([
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation((90, 90))], p=0.5),
    T.RandomApply([T.ColorJitter(0.2, 0.2, 0.1, 0.05)], p=0.5),
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
eval_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

train_ds = HistologicalImageDataset(f"{CFG['data_dir']}/train", transform=train_tf)
val_ds   = HistologicalImageDataset(f"{CFG['data_dir']}/val",   transform=eval_tf)
test_ds  = HistologicalImageDataset(f"{CFG['data_dir']}/test",  transform=eval_tf)

counts  = np.bincount(train_ds.labels)
weights = torch.from_numpy((1.0 / counts)[train_ds.labels]).double()
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"],
          pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, sampler=sampler, drop_last=False, **kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **kw)

CLASS_NAMES = train_ds.class_names
N_CLASSES   = len(CLASS_NAMES)
print(f"Classi: {CLASS_NAMES}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


Loading from /data/BREAKHIS/train...


Loading dataset from disk:   0%|          | 0/29 [00:00<?, ?it/s]

Loaded 25880 samples, 8 classes
Class distribution:
  adenosis: 1139 (4.4%)
  fibroadenoma: 3685 (14.2%)
  phyllodes_tumor: 1419 (5.5%)
  tubular_adenoma: 1642 (6.3%)
  ductal_carcinoma: 11717 (45.3%)
  lobular_carcinoma: 1927 (7.4%)
  mucinous_carcinoma: 2446 (9.5%)
  papillary_carcinoma: 1905 (7.4%)
Loading from /data/BREAKHIS/val...
Loaded 6832 samples, 8 classes
Class distribution:
  adenosis: 541 (7.9%)
  fibroadenoma: 692 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 601 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 601 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 448 (6.6%)
Loading from /data/BREAKHIS/test...
Loaded 6833 samples, 8 classes
Class distribution:
  adenosis: 540 (7.9%)
  fibroadenoma: 693 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 602 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 602 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 447 (6.5%)
Classi: ['adenosis', 'fibroadenoma',

In [5]:
class UNILoRAClassifier(nn.Module):
    def __init__(self, n_classes, dropout=0.1):
        super().__init__()
        backbone = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True,
                                      init_values=1e-5, dynamic_img_size=True)
        lora_config = LoraConfig(r=8, lora_alpha=32,
            target_modules=["qkv","proj","fc1","fc2"], lora_dropout=0.1, bias="none")
        self.backbone = LoraModel(backbone, lora_config, adapter_name="default")
        self.head = nn.Sequential(nn.LayerNorm(1024), nn.Dropout(dropout),
                                   nn.Linear(1024, n_classes))
    def forward(self, x):
        return self.head(self.backbone(x))

model = UNILoRAClassifier(N_CLASSES).to(device)
ckpt  = torch.load(CFG["output_dir"] / "best_model.pt", map_location=device)
model.load_state_dict(ckpt, strict=False)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print("Classificatore caricato")

Classificatore caricato


In [6]:
# Layer da raccogliere come target (curriculum + finale)
all_target_layers = sorted(set(
    list(CFG["curriculum_steps"]) + [CFG["layer_target"]]
))
# Layer da raccogliere come sorgente
all_source_layers = sorted(set(
    CFG["layers_source"] + list(CFG["curriculum_steps"])
))

def collect_and_save_dataset(model, loaders_dict, device,
                              layers_source, all_target_layers, save_path):
    with h5py.File(save_path, 'w') as f:
        for split_name, loader in loaders_dict.items():
            print(f"\nRaccolta {split_name}...")
            n_total   = len(loader.dataset)
            n_patches = 196
            embed_dim = 1024

            grp      = f.create_group(split_name)
            ds_label = grp.create_dataset("labels",
                shape=(n_total,), maxshape=(None,), dtype='i4', compression="gzip")
            # Dataset per ogni layer target (curriculum + finale)
            ds_attns = {t: grp.create_dataset(f"attn_layer{t}",
                shape=(n_total, n_patches), maxshape=(None, n_patches),
                dtype='f2', compression="gzip", chunks=(64, n_patches))
                for t in all_target_layers}
            ds_embs  = {l: grp.create_dataset(f"emb_layer{l}",
                shape=(n_total, n_patches, embed_dim),
                maxshape=(None, n_patches, embed_dim),
                dtype='f2', compression="gzip", chunks=(64, n_patches, embed_dim))
                for l in layers_source}

            orig  = {}
            cache = {}

            def make_hook(idx):
                def fwd(self, x):
                    B, N, C = x.shape
                    qkv  = self.qkv(x).reshape(B,N,3,self.num_heads,
                                     self.head_dim).permute(2,0,3,1,4)
                    q, k, v = qkv.unbind(0)
                    q, k   = self.q_norm(q), self.k_norm(k)
                    attn   = (q @ k.transpose(-2,-1) * self.scale).softmax(-1)
                    if idx in layers_source:
                        cache[f"emb_{idx}"] = x[:,1:].detach().cpu().half()
                    if idx in all_target_layers:
                        cache[f"attn_{idx}"] = attn[:,:,0,1:].mean(1).detach().cpu().half()
                    x = (self.attn_drop(attn) @ v).transpose(1,2).reshape(B,N,C)
                    return self.proj_drop(self.proj(x))
                return fwd

            for i, block in enumerate(model.backbone.model.blocks):
                orig[i] = block.attn.forward
                block.attn.forward = types.MethodType(make_hook(i), block.attn)

            FLUSH_EVERY = 16
            buf_labels  = []
            buf_attns   = {t: [] for t in all_target_layers}
            buf_embs    = {l: [] for l in layers_source}

            def flush(ptr):
                if not buf_labels: return ptr
                B = sum(len(x) for x in buf_labels)
                ds_label[ptr:ptr+B] = np.concatenate(buf_labels)
                for t in all_target_layers:
                    ds_attns[t][ptr:ptr+B] = torch.cat(buf_attns[t]).numpy()
                for l in layers_source:
                    ds_embs[l][ptr:ptr+B] = torch.cat(buf_embs[l]).numpy()
                buf_labels.clear()
                for t in all_target_layers: buf_attns[t].clear()
                for l in layers_source: buf_embs[l].clear()
                return ptr + B

            ptr = 0
            with torch.no_grad():
                for i_batch, (imgs, labels) in enumerate(tqdm(loader, desc=split_name)):
                    cache.clear()
                    model(imgs.to(device))
                    buf_labels.append(labels.numpy())
                    for t in all_target_layers:
                        buf_attns[t].append(cache[f"attn_{t}"])
                    for l in layers_source:
                        buf_embs[l].append(cache[f"emb_{l}"])
                    if (i_batch + 1) % FLUSH_EVERY == 0:
                        ptr = flush(ptr)

            ptr = flush(ptr)
            for i, block in enumerate(model.backbone.model.blocks):
                block.attn.forward = orig[i]
            if ptr < n_total:
                ds_label.resize(ptr, axis=0)
                for t in all_target_layers: ds_attns[t].resize(ptr, axis=0)
                for l in layers_source: ds_embs[l].resize(ptr, axis=0)
            print(f"  {split_name}: {ptr} sample salvati")

if not CFG["dataset_cache"].exists():
    collect_and_save_dataset(model,
        {"train": train_loader, "val": val_loader, "test": test_loader},
        device, all_source_layers, all_target_layers, CFG["dataset_cache"])
else:
    print(f"Cache trovata: {CFG['dataset_cache']}")
    with h5py.File(CFG["dataset_cache"], 'r') as f:
        for split in f.keys():
            print(f"  {split}: {len(f[split]['labels'])} sample")
            # Verifica che tutti i layer curriculum siano presenti
            missing = [t for t in all_target_layers
                       if f"attn_layer{t}" not in f[split]]
            if missing:
                print(f"  ⚠️  Layer mancanti: {missing} — riesegui la raccolta")


Cache trovata: /data/data_cache/BREAKHIS_forecaster_dataset.h5
  test: 6833 sample
  train: 25880 sample
  val: 6832 sample


In [7]:
class H5ForecastDataset(torch.utils.data.Dataset):
    def __init__(self, h5_path, split, layer_source, layer_target):
        self.h5_path      = str(h5_path)
        self.split        = split
        self.layer_source = layer_source
        self.layer_target = layer_target
        self._file        = None
        with h5py.File(h5_path, 'r') as f:
            self.length = len(f[split]["labels"])

    def _get_file(self):
        if self._file is None:
            self._file = h5py.File(self.h5_path, 'r', swmr=True)
        return self._file

    def __len__(self): return self.length

    def __getitem__(self, idx):
        f   = self._get_file()[self.split]
        emb    = torch.from_numpy(f[f"emb_layer{self.layer_source}"][idx]).float()
        target = torch.from_numpy(f[f"attn_layer{self.layer_target}"][idx]).float()
        return emb, target, int(f["labels"][idx])


In [8]:
class AttentionForecaster(nn.Module):
    def __init__(self, embed_dim=1024, hidden=256, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj  = nn.Linear(embed_dim, hidden)
        self.cls_query   = nn.Parameter(torch.randn(1,1,hidden)*0.02)
        self.self_attn   = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=hidden, nhead=n_heads,
                dim_feedforward=hidden*2, dropout=dropout,
                batch_first=True, norm_first=True) for _ in range(n_layers)])
        self.cross_attn  = nn.ModuleList([
            nn.MultiheadAttention(hidden, n_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers)])
        self.cross_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.norm        = nn.LayerNorm(hidden)
        self.score_head  = nn.Sequential(
            nn.Linear(hidden*2,128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128,1))

    def forward(self, x):
        B, N, D = x.shape
        x = self.input_proj(x)
        for sa in self.self_attn: x = sa(x)
        cls = self.cls_query.expand(B,-1,-1)
        for ca, norm in zip(self.cross_attn, self.cross_norms):
            cls_out, _ = ca(cls, x, x); cls = norm(cls + cls_out)
        x_n = self.norm(x)
        return self.score_head(torch.cat([x_n, cls.expand(-1,N,-1)], dim=-1)).squeeze(-1).softmax(-1)


In [9]:
def make_loaders(h5_path, layer_source, layer_target):
    kw = dict(batch_size=64, num_workers=4, pin_memory=True, persistent_workers=False)
    return (
        DataLoader(H5ForecastDataset(h5_path,"train",layer_source,layer_target),
                   shuffle=True, **kw),
        DataLoader(H5ForecastDataset(h5_path,"val",  layer_source,layer_target),
                   shuffle=False, **kw),
        DataLoader(H5ForecastDataset(h5_path,"test", layer_source,layer_target),
                   shuffle=False, **kw),
    )


def run_epoch(forecaster, loader, opt, device, cfg, train=True):
    forecaster.train() if train else forecaster.eval()
    total_kl, rho_list = 0., []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for emb, target, _ in tqdm(loader):
            emb, target = emb.to(device), target.to(device)
            pred = forecaster(emb)
            loss_kl  = F.kl_div((pred+1e-8).log(), target+1e-8, reduction='batchmean')
            loss_mse = F.mse_loss(pred, target)
            loss     = cfg["lambda_kl"] * loss_kl + cfg["lambda_mse"] * loss_mse
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(forecaster.parameters(), 1.0)
                opt.step()
            total_kl += loss_kl.item()
            if not train:
                for b in range(len(emb)):
                    rho, _ = spearmanr(pred[b].detach().cpu().numpy(),
                                       target[b].cpu().numpy())
                    rho_list.append(rho)
    return total_kl / len(loader), np.nanmean(rho_list) if rho_list else float('nan')


def train_with_curriculum(layer_source, cfg, device):
    run_name = f"curriculum_src{layer_source:02d}_tgt{cfg['layer_target']:02d}"
    print(f"\n{'='*60}\n  {run_name}\n{'='*60}")

    wandb.init(
        project = cfg["wandb_project"],
        name    = run_name,
        config  = {**{k:v for k,v in cfg.items()
                      if not isinstance(v, Path)},
                   "layer_source": layer_source,
                   "curriculum": True},
        tags    = [f"src{layer_source}", "curriculum", DATASET_NAME],
        reinit  = True,
    )

    forecaster = AttentionForecaster(
        embed_dim=1024, hidden=cfg["hidden"],
        n_heads=cfg["n_heads"], n_layers=cfg["n_layers"],
        dropout=cfg["dropout"]).to(device)

    # LR più alto nelle fasi curriculum, poi abbassato
    # (analogo al warmup usato da Li et al.)
    opt_curriculum = torch.optim.AdamW(forecaster.parameters(),
                                        lr=cfg["lr"]*3, weight_decay=cfg["weight_decay"])
    opt_final      = torch.optim.AdamW(forecaster.parameters(),
                                        lr=cfg["lr"],   weight_decay=cfg["weight_decay"])
    sched_final    = torch.optim.lr_scheduler.CosineAnnealingLR(
                        opt_final, T_max=cfg["epochs"])

    save_path   = cfg["forecaster_dir"] / f"forecaster_{run_name}.pt"
    best_val_kl = float('inf')
    best_val_rho= -1.
    global_step = 0

    # ── FASE CURRICULUM ──────────────────────────────
    # Ordine: dai layer vicini all'output verso layer_source
    # Tutti predicono attn @ layer_target (23)
    # ma ricevono embedding da layer_intermedio sempre più lontano
    curriculum_targets = sorted(cfg["curriculum_steps"], reverse=True)

    for cur_src in curriculum_targets:
        # Salta se cur_src <= layer_source (non ha senso)
        if cur_src <= layer_source:
            continue

        print(f"\n  Curriculum: emb@{cur_src} → attn@{cfg['layer_target']}")
        tr_ld, vl_ld, _ = make_loaders(cfg["dataset_cache"],
                                        cur_src, cfg["layer_target"])

        for ep in range(cfg["curriculum_epochs_per_step"]):
            tr_kl, _ = run_epoch(forecaster, tr_ld, opt_curriculum, device, cfg, train=True)
            vl_kl, vl_rho = run_epoch(forecaster, vl_ld, None, device, cfg, train=False)
            wandb.log({
                "curriculum_src" : cur_src,
                "train/kl"       : tr_kl,
                "val/kl"         : vl_kl,
                "val/rho"        : vl_rho,
                "phase"          : "curriculum",
                "global_step"    : global_step,
            })
            global_step += 1
        print(f"    val_kl={vl_kl:.4f} val_ρ={vl_rho:.3f}")

    # ── FASE FINALE su layer_source reale ────────────
    print(f"\n  Fase finale: emb@{layer_source} → attn@{cfg['layer_target']}")
    tr_ld, vl_ld, ts_ld = make_loaders(cfg["dataset_cache"],
                                        layer_source, cfg["layer_target"])

    for epoch in range(cfg["epochs"]):
        tr_kl, _ = run_epoch(forecaster, tr_ld, opt_final, device, cfg, train=True)
        vl_kl, vl_rho = run_epoch(forecaster, vl_ld, None, device, cfg, train=False)
        sched_final.step()
        wandb.log({
            "curriculum_src" : layer_source,
            "train/kl"       : tr_kl,
            "val/kl"         : vl_kl,
            "val/rho"        : vl_rho,
            "lr"             : sched_final.get_last_lr()[0],
            "phase"          : "final",
            "global_step"    : global_step,
            "epoch"          : epoch+1,
        })
        global_step += 1
        if vl_kl < best_val_kl:
            best_val_kl  = vl_kl
            best_val_rho = vl_rho
            torch.save(forecaster.state_dict(), save_path)
        if (epoch+1) % 5 == 0:
            print(f"    Ep{epoch+1:02d} | kl={vl_kl:.4f} ρ={vl_rho:.3f} best_ρ={best_val_rho:.3f}")

    # ── TEST ─────────────────────────────────────────
    forecaster.load_state_dict(torch.load(save_path))
    forecaster.eval()
    test_rho_f, test_rho_n = [], []
    with h5py.File(cfg["dataset_cache"], 'r') as f_h5:
        emb_all    = torch.from_numpy(f_h5["test"][f"emb_layer{layer_source}"][:]).float()
        target_all = torch.from_numpy(f_h5["test"][f"attn_layer{cfg['layer_target']}"][:]).float()

    with torch.no_grad():
        for emb, target in DataLoader(TensorDataset(emb_all, target_all),
                                       batch_size=64, shuffle=False):
            pred = forecaster(emb.to(device)).cpu()
            for b in range(len(emb)):
                t = target[b].numpy()
                rho_f, _ = spearmanr(pred[b].numpy(), t)
                rho_n, _ = spearmanr(emb[b].norm(dim=-1).numpy(), t)
                test_rho_f.append(rho_f); test_rho_n.append(rho_n)

    test_rho_f = np.nanmean(test_rho_f)
    test_rho_n = np.nanmean(test_rho_n)
    wandb.log({"test/rho_forecaster": test_rho_f,
               "test/rho_token_norm": test_rho_n,
               "test/delta"         : test_rho_f - test_rho_n,
               "best_val_rho"       : best_val_rho})
    print(f"\n  Test ρ forecaster: {test_rho_f:.3f} | baseline: {test_rho_n:.3f} "
          f"| Δ={test_rho_f-test_rho_n:+.3f}")
    wandb.finish()

    return {"layer_source": layer_source, "test_rho_forecaster": test_rho_f,
            "test_rho_token_norm": test_rho_n, "best_val_rho": best_val_rho,
            "best_val_kl": best_val_kl}


In [ ]:
all_results = []
for layer_source in CFG["layers_source"]:
    result = train_with_curriculum(layer_source, CFG, device)
    all_results.append(result)


  curriculum_src02_tgt23


wandb: Currently logged in as: vincenzo-civale (vincenzo-civale-universi-degli-studi-di-firenze) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



  Curriculum: emb@20 → attn@23


  0%|          | 0/405 [00:00<?, ?it/s]

  0%|          | 0/107 [00:00<?, ?it/s]

...

  0%|          | 0/107 [00:00<?, ?it/s]

    val_kl=-0.2145 val_ρ=0.805

  Fase finale: emb@4 → attn@23


  0%|          | 0/405 [00:00<?, ?it/s]

  0%|          | 0/107 [00:00<?, ?it/s]

  0%|          | 0/405 [00:00<?, ?it/s]

In [ ]:
layers   = [r["layer_source"]        for r in all_results]
rho_fore = [r["test_rho_forecaster"] for r in all_results]
rho_norm = [r["test_rho_token_norm"] for r in all_results]

fig, ax = plt.subplots(figsize=(11,5))
ax.plot(layers, rho_fore, marker='o', lw=2, color='tomato',
        label="AttentionForecaster + Curriculum")
ax.plot(layers, rho_norm, marker='s', lw=2, color='gray',
        ls='--', label="Token norm baseline")
ax.fill_between(layers, rho_norm, rho_fore, alpha=0.15, color='tomato')
ax.set_xlabel("Layer sorgente", fontsize=13)
ax.set_ylabel("Spearman ρ (test)", fontsize=13)
ax.set_title(f"Curriculum Forecaster — {DATASET_NAME}\n"
             f"target=layer {CFG['layer_target']}", fontsize=14)
ax.legend(fontsize=11); ax.grid(alpha=0.3); ax.set_xticks(layers)
plt.tight_layout()
out = CFG["forecaster_dir"] / "curriculum_rho_vs_layer.png"
plt.savefig(out, dpi=150); plt.show()
print(f"Salvato in {out}")

print(f"\n{'Layer':>8} {'ρ Curriculum':>14} {'ρ Baseline':>12} {'Δ':>8}")
print("─"*46)
for r in all_results:
    d = r["test_rho_forecaster"] - r["test_rho_token_norm"]
    print(f"{r['layer_source']:>8} {r['test_rho_forecaster']:>14.3f} "
          f"{r['test_rho_token_norm']:>12.3f} {d:>+8.3f}")
